# Configuration and imports

In [0]:
# Feature 1.3: Safe reruns for Toronto Airbnb Bronze ingestion.
#
# This notebook reads the raw Toronto file from the Volume,
# adds provenance metadata, replaces only the matching city/snapshot slice,
# and records the run in an ingestion-control table.

from datetime import UTC, datetime

from pyspark.sql import functions as F

from src.ingestion.control import should_load
from src.ingestion.provenance import build_provenance_columns, build_run_id

# Real Databricks environment values.
CATALOG = "trippilot_dev"
BRONZE_SCHEMA = "bronze"

# Source details for the Toronto Airbnb snapshot.
SOURCE = "insideairbnb"
CITY = "toronto"
SNAPSHOT_DATE = "2026-09-16"
RAW_PATH = (
    "/Volumes/trippilot_dev/bronze/source_files/"
    "insideairbnb/toronto/listings.csv.gz"
)

# Destination Bronze table.
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.airbnb_listings_bronze"

# Operational audit table used to prevent duplicate loads.
CONTROL_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.airbnb_ingestion_control"

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Control table: {CONTROL_TABLE}")

# Check whether this run should proceed

In [0]:
# Read prior control records for this exact city and snapshot.
#
# If a previous run already succeeded, we stop before reading or writing data.
# This is the main duplicate-prevention check.

existing_controls = [
    row.asDict()
    for row in spark.table(CONTROL_TABLE)
    .filter(
        (F.col("city") == CITY)
        & (F.col("snapshot_date") == SNAPSHOT_DATE)
    )
    .collect()
]

# Ask the reusable helper whether this city/snapshot needs loading.
should_run = should_load(
    existing_controls=existing_controls,
    city=CITY,
    snapshot_date=SNAPSHOT_DATE,
)

print(f"Existing control records: {len(existing_controls)}")
print(f"Should run ingestion: {should_run}")

# Stop cleanly if this exact city/snapshot has already loaded successfully.
assert should_run, (
    f"Skipping ingestion: {CITY} / {SNAPSHOT_DATE} already loaded successfully."
)

# Start the control record

In [0]:
# Create a unique ID for this pipeline execution.
run_id = build_run_id()

# Record when this run started, in UTC.
started_at_utc = datetime.now(UTC).isoformat()

# Insert a "running" record before doing any data work.
# If a later cell fails, this record can be updated to "failed".
spark.sql(f"""
    INSERT INTO {CONTROL_TABLE}
    VALUES (
        '{CITY}',
        '{SNAPSHOT_DATE}',
        '{run_id}',
        'running',
        0,
        '{started_at_utc}',
        NULL,
        NULL
    )
""")

print(f"Started ingestion run: {run_id}")

# Read the raw Volume file

In [0]:
# Read the raw compressed CSV from the Volume.
#
# inferSchema=False keeps source values as strings in Bronze.
# Type cleaning and casting belong to Silver, not Bronze.

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(RAW_PATH)
)

# Count input rows now so we can compare them with Bronze output later.
raw_row_count = raw_df.count()

print(f"Raw rows read: {raw_row_count}")
display(raw_df.limit(5))

# Add provenance metadata

In [0]:
# Build provenance metadata for this specific ingestion run.
#
# These fields make every Bronze row traceable to its source file,
# city, snapshot date, and pipeline execution.

provenance = build_provenance_columns(
    source=SOURCE,
    city=CITY,
    snapshot_date=SNAPSHOT_DATE,
    raw_path=RAW_PATH,
    run_id=run_id,
)

# Add metadata columns to every raw Airbnb record.
# The underscore prefix separates pipeline metadata from original Airbnb fields.
bronze_df = (
    raw_df
    .withColumn("_source", F.lit(provenance["source"]))
    .withColumn("_city", F.lit(provenance["city"]))
    .withColumn("_snapshot_date", F.lit(provenance["snapshot_date"]))
    .withColumn("_raw_path", F.lit(provenance["raw_path"]))
    .withColumn("_run_id", F.lit(provenance["run_id"]))
    .withColumn("_ingested_at_utc", F.lit(provenance["ingested_at_utc"]))
)

display(bronze_df.limit(5))

# Replace only the snapshot slice

In [0]:
# Replace only this city/snapshot slice.
#
# This is safer than overwriting the whole Bronze table because future
# cities and future snapshots remain untouched.

spark.sql(f"""
    DELETE FROM {BRONZE_TABLE}
    WHERE _city = '{CITY}'
      AND _snapshot_date = '{SNAPSHOT_DATE}'
""")

# Append the newly read Toronto rows.
# Because we deleted only the matching slice immediately before this write,
# rerunning the same snapshot will not create duplicates.
(
    bronze_df
    .write
    .mode("append")
    .saveAsTable(BRONZE_TABLE)
)

# Count what was actually written for verification.
bronze_row_count = (
    spark.table(BRONZE_TABLE)
    .filter(
        (F.col("_city") == CITY)
        & (F.col("_snapshot_date") == SNAPSHOT_DATE)
    )
    .count()
)

print(f"Bronze rows written for {CITY} / {SNAPSHOT_DATE}: {bronze_row_count}")

# Record success and verify

In [0]:
# Record completion time and mark this run successful.
completed_at_utc = datetime.now(UTC).isoformat()

spark.sql(f"""
    UPDATE {CONTROL_TABLE}
    SET status = 'success',
        row_count = {bronze_row_count},
        completed_at_utc = '{completed_at_utc}',
        error_message = NULL
    WHERE run_id = '{run_id}'
""")

# Verify that this city/snapshot contains one run and the expected row count.
verification_df = (
    spark.table(BRONZE_TABLE)
    .filter(
        (F.col("_city") == CITY)
        & (F.col("_snapshot_date") == SNAPSHOT_DATE)
    )
    .agg(
        F.count("*").alias("bronze_row_count"),
        F.countDistinct("_run_id").alias("distinct_run_ids"),
    )
)

display(verification_df)

# The Bronze row count must match the raw input row count.
assert bronze_row_count == raw_row_count, (
    f"Row-count mismatch: raw={raw_row_count}, bronze={bronze_row_count}"
)

print("Bronze ingestion verification passed.")

In [0]:
%sql
SELECT
    _city,
    _snapshot_date,
    COUNT(*) AS row_count,
    COUNT(DISTINCT _run_id) AS run_count
FROM trippilot_dev.bronze.airbnb_listings_bronze
GROUP BY _city, _snapshot_date;